# День 3 — Attention-матрицы и визуализация

Цель: понять attention weights и научиться их визуализировать.
Используем токенизатор из Дня 1 и модель из Дня 2.

## Задача 1: загрузка модели с attention

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, output_attentions=True)
model.eval()

text = "The amazing movie won many awards"
tokens = tokenizer(text, return_tensors="pt")

## Задача 2: получение attention-весов

In [ ]:
with torch.no_grad():
    outputs = model(**tokens)

print(type(outputs.attentions))
print(f'Количество слоёв: {len(outputs.attentions)}')
print(f'Форма attention для слоя 0: {outputs.attentions[0].shape}')

attention = outputs.attentions[0]
print(f'Attention shape: {attention.shape}')

attn_single = attention[0, 0]
print(f'Single head shape: {attn_single.shape}')

## Задача 3: визуализация attention

`matplotlib` и `seaborn` уже есть в зависимостях проекта. Функция
`visualize_attention` лежит в `sentiment_lab.attention`: она сохраняет
heatmap в `outputs/` (папка в `.gitignore`) и показывает его.

In [ ]:
from sentiment_lab.attention import visualize_attention

In [ ]:
visualize_attention(tokens, outputs.attentions, tokenizer, layer=0, head=0)

## Задача 4: внимание в разных слоях

Первый (0), средний (3) и последний (5) слои DistilBERT.

In [ ]:
for layer in [0, 3, 5]:
    path = visualize_attention(tokens, outputs.attentions, tokenizer, layer=layer, head=0)
    print(path)

## Задача 5: разные головы одного слоя

У DistilBERT 8 голов; разные головы фокусируются на разных связях.

In [ ]:
for head in range(model.config.num_attention_heads):
    visualize_attention(tokens, outputs.attentions, tokenizer, layer=0, head=head)

## Задача 6: внимание к ключевому слову

Смотрим, на какие токены смотрит `terrible` в последнем слое.

In [ ]:
from sentiment_lab.attention import attention_for_token, get_attention

text = "This movie was absolutely terrible and I hated it"
tokens, attentions = get_attention(text, tokenizer, model)
visualize_attention(tokens, attentions, tokenizer, layer=5, head=0)

for token, weight in attention_for_token(
    tokens, attentions, tokenizer, query="terrible", layer=5, head=0, top_k=5
):
    print(f'{token}: {weight:.3f}')